# Query Optimization

In [163]:
# Imports
import time
import json
from pathlib import Path

import duckdb
import pandas as pd

In [164]:
# Config
DB_PATH = "/workspace/data/lecture.duckdb"
PROFILE_DIR = Path("/workspace/data/profiles")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

In [165]:
# Connect to DuckDB
con = duckdb.connect(DB_PATH)
con.sql("PRAGMA threads=4")
print("Connected to", DB_PATH)

Connected to /workspace/data/lecture.duckdb


In [166]:
# Helper functions

# Given a SQL query, print the plan
def show_plan(sql: str, analyze: bool = False):
    prefix = "EXPLAIN ANALYZE" if analyze else "EXPLAIN"
    plan_text = con.sql(f"{prefix} {sql}").fetchone()[1]
    print(plan_text)

# Given a SQL query, benchmark it by running it `runs` times.
# Returns average execution time in milliseconds.
def benchmark(sql: str, runs: int = 3, warmup: int = 1) -> float:
    for _ in range(warmup):
        con.sql(sql).fetchall()

    times_ms = []
    for _ in range(runs):
        t0 = time.perf_counter()
        con.sql(sql).fetchall()
        elapsed = (time.perf_counter() - t0) * 1000
        times_ms.append(elapsed)

    return float(sum(times_ms) / len(times_ms))

# Given OFF/ON averages, print speedup and simple timing lines.
def report_off_on(
    off_avg: float,
    on_avg: float,
    off_label: str = "optimizer_off",
    on_label: str = "optimizer_on",
    speedup_title: str = "Relative speedup (OFF/ON, avg)",
) -> None:
    speedup = off_avg / on_avg if on_avg > 0 else float("inf")
    print(f"\n{speedup_title}: {speedup:.2f}x")
    print(f"{off_label}: {off_avg:.2f} ms")
    print(f"{on_label}:  {on_avg:.2f} ms")


## Motivational Example: same query, optimizer OFF vs ON

In [167]:
motivational_query = """
SELECT COUNT(*)
FROM range(1, 1000001) a(i)
JOIN range(1, 1000001) b(j) ON i = j
WHERE i BETWEEN 1000 AND 50000
"""

# Optimizer OFF first
con.sql("PRAGMA disable_optimizer")
off_avg = benchmark(motivational_query)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
on_avg = benchmark(motivational_query)

report_off_on(off_avg, on_avg)


Relative speedup (OFF/ON, avg): 13.49x
optimizer_off: 55.67 ms
optimizer_on:  4.13 ms


## 1) Create an artificial dataset

Schema overview:

- `customers(customer_id, city, segment, signup_date)`
- `products(product_id, category, brand, list_price)`
- `orders(order_id, customer_id, order_date, status)`
- `order_items(order_id, line_no, product_id, quantity, unit_price)`


In [168]:
# Clean rebuild for reproducibility
con.sql("DROP TABLE IF EXISTS order_items")
con.sql("DROP TABLE IF EXISTS orders")
con.sql("DROP TABLE IF EXISTS products")
con.sql("DROP TABLE IF EXISTS customers")

con.sql("""
CREATE TABLE customers AS
SELECT
    i AS customer_id,
    CASE
        WHEN i % 8 = 0 THEN 'Istanbul'
        WHEN i % 8 = 1 THEN 'Berlin'
        WHEN i % 8 = 2 THEN 'Paris'
        WHEN i % 8 = 3 THEN 'London'
        WHEN i % 8 = 4 THEN 'Madrid'
        WHEN i % 8 = 5 THEN 'Rome'
        WHEN i % 8 = 6 THEN 'Amsterdam'
        ELSE 'Vienna'
    END AS city,
    CASE
        WHEN i % 3 = 0 THEN 'Consumer'
        WHEN i % 3 = 1 THEN 'Small Business'
        ELSE 'Enterprise'
    END AS segment,
    DATE '2022-01-01' + CAST(i % 730 AS INTEGER) AS signup_date
FROM range(1, 20001) AS t(i)
""")

con.sql("""
CREATE TABLE products AS
SELECT
    i AS product_id,
    CASE
        WHEN i % 5 = 0 THEN 'Electronics'
        WHEN i % 5 = 1 THEN 'Home'
        WHEN i % 5 = 2 THEN 'Fashion'
        WHEN i % 5 = 3 THEN 'Sports'
        ELSE 'Books'
    END AS category,
    'Brand_' || CAST((i % 25) AS VARCHAR) AS brand,
    ROUND(10 + ((i * 17) % 3000) / 10.0, 2) AS list_price
FROM range(1, 501) AS t(i)
""")

con.sql("""
CREATE TABLE orders AS
SELECT
    i AS order_id,
    ((i * 37) % 20000) + 1 AS customer_id,
    DATE '2023-01-01' + CAST(i % 730 AS INTEGER) AS order_date,
    CASE
        WHEN i % 20 = 0 THEN 'cancelled'
        WHEN i % 5 = 0 THEN 'shipped'
        ELSE 'placed'
    END AS status
FROM range(1, 400001) AS t(i)
""")

con.sql("""
CREATE TABLE order_items AS
SELECT
    o.order_id,
    l.line_no,
    ((o.order_id * (l.line_no + 13)) % 500) + 1 AS product_id,
    ((o.order_id + l.line_no) % 4) + 1 AS quantity,
    ROUND(5 + ((o.order_id * (l.line_no + 7)) % 2500) / 10.0, 2) AS unit_price
FROM orders o
CROSS JOIN (SELECT i AS line_no FROM range(1, 4) AS t(i)) l
""")

# Gather initial stats
con.sql("ANALYZE")

print("Schema and data are ready.")

Schema and data are ready.


In [169]:
# Quick sanity checks
counts = con.sql("""
SELECT 'customers' AS table_name, COUNT(*) AS rows FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
""").df()

counts

,table_name,rows
0,customers,20000
1,products,500
2,orders,400000
3,order_items,1200000


In [170]:
# For a sanity check, select the first 3 rows of each table
con.sql("SELECT * FROM customers LIMIT 3").df()

,customer_id,city,segment,signup_date
0,1,Berlin,Small Business,2022-01-02
1,2,Paris,Enterprise,2022-01-03
2,3,London,Consumer,2022-01-04


In [171]:
con.sql("SELECT * FROM products LIMIT 3").df()

,product_id,category,brand,list_price
0,1,Home,Brand_1,11.7
1,2,Fashion,Brand_2,13.4
2,3,Sports,Brand_3,15.1


In [172]:
con.sql("SELECT * FROM orders LIMIT 3").df()

,order_id,customer_id,order_date,status
0,1,38,2023-01-02,placed
1,2,75,2023-01-03,placed
2,3,112,2023-01-04,placed


In [173]:
con.sql("SELECT * FROM order_items LIMIT 3").df()

,order_id,line_no,product_id,quantity,unit_price
0,1,1,15,3,5.8
1,2,1,29,4,6.6
2,3,1,43,1,7.4


## 1) Expression rewriting


### Expression rewriting: Example 1

In [174]:
query_expr_raw = """
SELECT COUNT(*)
FROM orders o, customers c
WHERE ((o.order_id + 0) = 345678)
  AND ((o.customer_id * 1) > 0)
  AND ((o.status = 'placed') OR TRUE)
  AND ((c.city IS NULL) OR (c.city IS NOT NULL))
  AND (10 - 10 = 0)
  AND TRUE
"""

# Optimizer OFF first
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: raw expression-heavy query with optimizer OFF ===")
show_plan(query_expr_raw, analyze=False)
expr_off_avg = benchmark(query_expr_raw)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: raw expression-heavy query with optimizer ON ===")
show_plan(query_expr_raw, analyze=False)
expr_on_avg = benchmark(query_expr_raw)

report_off_on(
    expr_off_avg,
    expr_on_avg,
    off_label="expression_raw_off",
    on_label="expression_raw_on",
    speedup_title="Relative speedup for expression rewriter demo (OFF/ON, avg)",
)

=== EXPLAIN: raw expression-heavy query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│  (((order_id + CAST(0 AS  │
│  BIGINT)) = CAST(345678 AS│
│       BIGINT)) AND (      │
│  (customer_id * CAST(1 AS │
│    BIGINT)) > CAST(0 AS   │
│   BIGINT)) AND ((status = │
│  CAST('placed' AS VARCHAR)│
│ ) OR CAST('t' AS BOOLEAN))│
│   AND ((city IS NULL) OR  │
│ (city IS NOT NULL)) AND ( │
│   (10 - 10) = CAST(0 AS   │
│  INTEGER)) AND CAST('t' AS│
│          BOOLEAN))        │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       CROSS_PRODUCT       ├──────────────┐
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│       

### Expression rewriting: Example 2 (Overlapping ranges)

In [175]:
query_between_overlap = """
SELECT COUNT(*)
FROM range(1, 1000001) t(x)
WHERE x BETWEEN 50 AND 150
  AND x BETWEEN 100 AND 200
"""

# Optimizer OFF
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: overlapping BETWEEN with optimizer OFF ===")
show_plan(query_between_overlap, analyze=False)
between_off_avg = benchmark(query_between_overlap)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: overlapping BETWEEN with optimizer ON ===")
show_plan(query_between_overlap, analyze=False)
between_on_avg = benchmark(query_between_overlap)

report_off_on(
    between_off_avg,
    between_on_avg,
)

=== EXPLAIN: overlapping BETWEEN with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ ((x >= CAST(50 AS BIGINT))│
│    AND (x >= CAST(100 AS  │
│   BIGINT)) AND (x <= CAST │
│ (150 AS BIGINT)) AND (x <=│
│    CAST(200 AS BIGINT)))  │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           RANGE           │
│    ────────────────────   │
│      Function: RANGE      │
│                           │
│          ~0 rows          │
└───────────────────────────┘


=== EXPLAIN: overlapping BETWEEN with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬──

### Expression rewriting: Example 3 (IN-clause rewriter) 


In [176]:
values_in = ", ".join(str(i) for i in range(1, 601))
query_in_clause = f"""
SELECT COUNT(*)
FROM orders
WHERE customer_id IN ({values_in})
"""

con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: large IN-clause query with optimizer OFF ===")
show_plan(query_in_clause, analyze=False)
in_off_avg = benchmark(query_in_clause)

con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: large IN-clause query with optimizer ON ===")
show_plan(query_in_clause, analyze=False)
in_on_avg = benchmark(query_in_clause)

report_off_on(
    in_off_avg,
    in_on_avg,
    off_label="in_clause_off",
    on_label="in_clause_on",
)

=== EXPLAIN: large IN-clause query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ (customer_id IN (CAST(1 AS│
│  BIGINT), CAST(2 AS BIGINT│
│ ), CAST(3 AS BIGINT), CAST│
│  (4 AS BIGINT), CAST(5 AS │
│  BIGINT), CAST(6 AS BIGINT│
│ ), CAST(7 AS BIGINT), CAST│
│  (8 AS BIGINT), CAST(9 AS │
│     BIGINT), CAST(10 AS   │
│     BIGINT), CAST(11 AS   │
│     BIGINT), CAST(12 AS   │
│     BIGINT), CAST(13 AS   │
│     BIGINT), CAST(14 AS   │
│     BIGINT), CAST(15 AS   │
│     BIGINT), CAST(16 AS   │
│     BIGINT), CAST(17 AS   │
│     BIGINT), CAST(18 AS   │
│     BIGINT), CAST(19 AS   │
│     BIGINT), CAST(20 AS   │
│     BIGINT), CAST(21 AS   │
│     BIGINT), CAST(22 AS   │
│     BIGINT), CAST(23 AS   │
│     BIGINT), CAST(24 AS   │
│     BIGINT

## 2) Filter pushdown

### Filter pushdown: Example 1

In [177]:
query_filter_pushdown = """
SELECT COUNT(*)
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-31'
  AND o.status = 'placed'
"""

# Optimizer OFF
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: filter pushdown with optimizer OFF ===")
show_plan(query_filter_pushdown, analyze=False)
filter_off_avg = benchmark(query_filter_pushdown)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: filter pushdown with optimizer ON ===")
show_plan(query_filter_pushdown, analyze=False)
filter_on_avg = benchmark(query_filter_pushdown)

report_off_on(
    filter_off_avg,
    filter_on_avg,
)

=== EXPLAIN: filter pushdown with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ ((order_date >= CAST('2024│
│   -01-01' AS DATE)) AND   │
│ (status = CAST('placed' AS│
│  VARCHAR)) AND (order_date│
│   <= CAST('2024-01-31' AS │
│           DATE)))         │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│    order_id = order_id    │              │
│                           │              │
│          ~0 rows          │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌───

## 3) Projection pushdown

### Projection pushdown: Example 1

In [178]:
query_projection_pushdown = """
SELECT SUM(oi.quantity) AS total_qty
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
"""

# Optimizer OFF
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: projection pushdown with optimizer OFF ===")
show_plan(query_projection_pushdown, analyze=False)
proj_off_avg = benchmark(query_projection_pushdown)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: projection pushdown with optimizer ON ===")
show_plan(query_projection_pushdown, analyze=False)
proj_on_avg = benchmark(query_projection_pushdown)

report_off_on(
    proj_off_avg,
    proj_on_avg,
)

=== EXPLAIN: projection pushdown with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│    Aggregates: sum(#0)    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│          quantity         │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│    order_id = order_id    │              │
│                           │              │
│          ~0 rows          │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│         SEQ_SCAN          ││         SEQ_SCAN          │
│    ────────────────────   ││    ────────────────────   │
│       Table: orders       ││  

## 5) TOP-N query optimization (ORDER BY + LIMIT)

### TOP-N query optimization: Example 1 

In [179]:
query_topn_demo = """
SELECT
    order_id,
    product_id,
    unit_price,
    quantity
FROM order_items
ORDER BY unit_price
LIMIT 20
"""

# Optimizer OFF 
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: Top-N query with optimizer OFF ===")
show_plan(query_topn_demo, analyze=False)
topn_demo_off_avg = benchmark(query_topn_demo)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: Top-N query with optimizer ON ===")
show_plan(query_topn_demo, analyze=False)
topn_demo_on_avg = benchmark(query_topn_demo)

report_off_on(
    topn_demo_off_avg,
    topn_demo_on_avg,
)

=== EXPLAIN: Top-N query with optimizer OFF ===
┌───────────────────────────┐
│           LIMIT           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│  lecture.main.order_items │
│      .unit_price ASC      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│     Table: order_items    │
│   Type: Sequential Scan   │
│                           │
│          ~0 rows          │
└───────────────────────────┘


=== EXPLAIN: Top-N query with optimizer ON ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_integ│
│     ral_bigint(#0, 1)     │
│__internal_decompress_integ│
│     ral_bigint(#1, 1)     │
│             #2            │
│__internal_decompress_integ│
│     ral_bigint(#3, 1)     │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌──

## 6) Constant-false subtree pruning 


### Constant-false subtree pruning : Example 1 

In [180]:
query_false_prune = """
SELECT COUNT(*)
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN customers c ON c.customer_id = o.customer_id
WHERE 1 = 0
"""

con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: constant-false query with optimizer OFF ===")
show_plan(query_false_prune, analyze=False)
false_off_avg = benchmark(query_false_prune)

con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: constant-false query with optimizer ON ===")
show_plan(query_false_prune, analyze=False)
false_on_avg = benchmark(query_false_prune)

report_off_on(
    false_off_avg,
    false_on_avg,
)

=== EXPLAIN: constant-false query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ (CAST(1 AS INTEGER) = CAST│
│      (0 AS INTEGER))      │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├───────────────────────────────────────────┐
│ customer_id = customer_id │                                           │
│                           │                                           │
│          ~0 rows          │                                           │
└─────────────┬─────────────┘                                           │
┌──

### Constant-false subtree pruning : Example 2

In [181]:
query_count_impossible = """
SELECT COUNT(*)
FROM orders
WHERE order_date < DATE '2024-01-01'
  AND order_date > DATE '2024-12-31'
"""

# Optimizer OFF
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: impossible COUNT query with optimizer OFF ===")
show_plan(query_count_impossible, analyze=False)
count_impossible_off_avg = benchmark(query_count_impossible)

# Optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: impossible COUNT query with optimizer ON ===")
show_plan(query_count_impossible, analyze=False)
count_impossible_on_avg = benchmark(query_count_impossible)

report_off_on(
    count_impossible_off_avg,
    count_impossible_on_avg,
)

=== EXPLAIN: impossible COUNT query with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│ ((order_date < CAST('2024 │
│   -01-01' AS DATE)) AND   │
│  (order_date > CAST('2024 │
│     -12-31' AS DATE)))    │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│       Table: orders       │
│   Type: Sequential Scan   │
│                           │
│          ~0 rows          │
└───────────────────────────┘


=== EXPLAIN: impossible COUNT query with optimizer ON ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└──────────

## 7) Join reordering 

### Join reordering : Example 1

In [182]:
query_join_reordering = """
SELECT COUNT(*)
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN customers c ON c.customer_id = o.customer_id
JOIN products p ON p.product_id = oi.product_id
WHERE c.city = 'Berlin'
  AND c.segment = 'Enterprise'
  AND p.category = 'Electronics'
  AND o.order_date BETWEEN DATE '2024-01-01' AND DATE '2024-03-31'
"""

# join-order optimizer disabled
con.sql("PRAGMA enable_optimizer")
con.sql("SET disabled_optimizers = 'join_order'")
print("=== EXPLAIN: join order optimizer DISABLED ===")
show_plan(query_join_reordering, analyze=False)
join_order_off_avg = benchmark(query_join_reordering)

# join-order optimizer enabled
con.sql("SET disabled_optimizers = ''")
print("\n=== EXPLAIN: join order optimizer ENABLED ===")
show_plan(query_join_reordering, analyze=False)
join_order_on_avg = benchmark(query_join_reordering)

# cleanup: restore defaults
con.sql("SET disabled_optimizers = ''")

report_off_on(
    join_order_off_avg,
    join_order_on_avg,
    off_label="join_order_disabled",
    on_label="join_order_enabled",
    speedup_title="Relative speedup for join reordering (disabled/enabled, avg)",
)

=== EXPLAIN: join order optimizer DISABLED ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├────────────────────────────────────────────────────────────────────────┐
│  product_id = product_id  │                                                                        │
│                           │                                                                        │
│          ~0 rows          │                                                                        │
└─────────────┬─────────────┘                                                                        │
┌─────────────┴─────────────┐                                                          ┌─────────────┴──────

## Query Optimizers are NOT perfect!

In [183]:
query_union_branches = """
SELECT COUNT(*)
FROM (
    SELECT customer_id
    FROM orders
    WHERE status = 'cancelled'

    UNION

    SELECT customer_id
    FROM orders
    WHERE status = 'shipped'

    UNION

    SELECT customer_id
    FROM orders
    WHERE status = 'placed'
) u
"""

# optimizer OFF
con.sql("PRAGMA disable_optimizer")
print("=== EXPLAIN: UNION branches on orders with optimizer OFF ===")
show_plan(query_union_branches, analyze=False)
union_off_avg = benchmark(query_union_branches)

# optimizer ON
con.sql("PRAGMA enable_optimizer")
print("\n=== EXPLAIN: UNION branches on orders with optimizer ON ===")
show_plan(query_union_branches, analyze=False)
union_on_avg = benchmark(query_union_branches)

report_off_on(
    union_off_avg,
    union_on_avg,
)

=== EXPLAIN: UNION branches on orders with optimizer OFF ===
┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│        count_star()       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           UNION           ├───────────────────────────────────────────┐
└─────────────┬─────────────┘                                           │
┌─────────────┴─────────────┐                             ┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │                             │         PROJECTION        │
│    ────────────────────   │                             │    ────────────────────   │
│         Groups: #0        │                             │        customer_id        │
│                  